**Классификация цветов Ирисов**

In [8]:
import Pkg
Pkg.add(["MLJ", "MLJBase", "MLJModels", "DataFrames", "CSV", "RDatasets", "NearestNeighborModels", "Plots", "StatsBase"])

     Project No packages added to or removed from `C:\Users\Lenovo\.julia\environments\v1.13\Project.toml`
    Manifest No packages added to or removed from `C:\Users\Lenovo\.julia\environments\v1.13\Manifest.toml`


In [9]:
using MLJ
using RDatasets
using DataFrames
using Plots
using Random

iris = dataset("datasets", "iris")
first(iris, 5)

Row,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
,Float64,Float64,Float64,Float64,Cat…
1,5.1,3.5,1.4,0.2,setosa
2,4.9,3.0,1.4,0.2,setosa
3,4.7,3.2,1.3,0.2,setosa
4,4.6,3.1,1.5,0.2,setosa
5,5.0,3.6,1.4,0.2,setosa


In [10]:
iris.Species = categorical(iris.Species)

schema(iris)

┌─────────────┬───────────────┬─────────────────────────────────┐
│ names       │ scitypes      │ types                           │
├─────────────┼───────────────┼─────────────────────────────────┤
│ SepalLength │ Continuous    │ Float64                         │
│ SepalWidth  │ Continuous    │ Float64                         │
│ PetalLength │ Continuous    │ Float64                         │
│ PetalWidth  │ Continuous    │ Float64                         │
│ Species     │ Multiclass{3} │ CategoricalValue{String, UInt8} │
└─────────────┴───────────────┴─────────────────────────────────┘


In [11]:
Random.seed!(69)

train, test = partition(iris, 0.8, shuffle=true, rng=42)

println("Обучающая выборка: $(nrow(train)) строк")
println("Тестовая выборка:  $(nrow(test)) строк")

Обучающая выборка: 120 строк
Тестовая выборка:  30 строк


In [12]:
KNNClassifier = @load KNNClassifier pkg=NearestNeighborModels

knn = KNNClassifier(K=3)

X = select(train, Not(:Species))
y = train.Species

mach = machine(knn, X, y)

fit!(mach)

import NearestNeighborModels ✔


┌ Info: For silent loading, specify `verbosity=0`. 
└ @ Main C:\Users\Lenovo\.julia\packages\MLJModels\bFCZ8\src\loading.jl:159
┌ Info: Training machine(KNNClassifier(K = 3, …), …).
└ @ MLJBase C:\Users\Lenovo\.julia\packages\MLJBase\x4XyA\src\machines.jl:537


trained Machine; caches model-specific representations of data
  model: KNNClassifier(K = 3, …)
  args: 
    1:	Source @927 ⏎ Table{AbstractVector{Continuous}}
    2:	Source @193 ⏎ AbstractVector{Multiclass{3}}


In [13]:
X_test = select(test, Not(:Species))
y_test = test.Species

y_pred = predict_mode(mach, X_test)

accuracy = MLJ.accuracy(y_pred, y_test)

println("Точность модели на тестовой выборке: $(round(accuracy * 100, digits=2))%")

Точность модели на тестовой выборке: 96.67%
